In [ ]:
# Setup
import math_toolkit
math_toolkit.activate()


# Radon inversion and missing data on a cat image

This notebook uses the cat bitmaps as functions on the flat torus $[0,1]^2$. The image samples are the finite model: their two-dimensional FFT gives the Fourier coefficients of the sampled torus function.

The point is to compare three reconstructions at the same cutoff $K$:

1. Fourier inversion truncated to $|n|<K$ and $|m|<K$.
2. Perfect Radon inversion, where the Fourier-slice theorem recovers exactly the same retained Fourier coefficients from exact Radon projection coefficients.
3. A lossy Radon-plus-Fourier inversion, where only finitely many projection directions and theta samples are kept.

The third reconstruction is intentionally incomplete. It is meant to show the high-frequency streaking and directional artifacts caused by missing Radon data, not to be a good image codec.


## Fourier slice theorem on the torus

For a primitive integer line direction $v=(p,q)$, define the torus Radon projection

$$
R_v f(x)=\int_0^1 f(x+t v)\,dt.
$$

The projection is constant along $v$. With quotient coordinate

$$
\theta=qx_1-px_2 \pmod 1,
$$

we can write $R_v f(x)=G_v(\theta)$. Its one-dimensional Fourier series is

$$
G_v(\theta)=\sum_{r\in\mathbb Z}\widehat f(r(q,-p))e^{2\pi i r\theta}.
$$

Thus each Fourier coefficient of a projection is one Fourier coefficient of the original image. If $(n,m)=d(a,b)$ with $(a,b)$ primitive, then the Radon direction $v=(-b,a)$ recovers $\widehat f(n,m)$ as the $d$-th Fourier coefficient of $G_v$.

In the perfect experiment below, this is just exact bookkeeping in Fourier space. In the lossy experiment, we only store projections for simple primitive normals and only store a finite theta resolution for each projection.


In [ ]:
from math import ceil, gcd
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import clear_output, display
from plotly.subplots import make_subplots


IMAGE_FILES = ["radon-cat-1.bmp", "radon-cat-2.bmp"]
DEFAULT_K = 80
MAX_K = 200
DEFAULT_L = 200
MAX_L = 1000
DEFAULT_NORMAL_BOUND = 24
DEFAULT_THETA_RESOLUTION = 160
DISPLAY_SIZE = 340


def md(text):
    """Display Markdown text in the notebook."""

    from IPython.display import Markdown

    display(Markdown(text))


def load_bmp_rgb(path):
    """Load the uncompressed BMP files used by this notebook."""

    data = Path(path).read_bytes()
    if data[:2] != b"BM":
        raise ValueError(f"{path} is not a BMP file.")

    pixel_offset = int.from_bytes(data[10:14], "little")
    width = int.from_bytes(data[18:22], "little", signed=True)
    signed_height = int.from_bytes(data[22:26], "little", signed=True)
    bits_per_pixel = int.from_bytes(data[28:30], "little")
    compression = int.from_bytes(data[30:34], "little")

    if width <= 0 or signed_height == 0:
        raise ValueError("Unsupported BMP dimensions.")
    height = abs(signed_height)
    row_stride = ((width * bits_per_pixel + 31) // 32) * 4
    rows = range(height - 1, -1, -1) if signed_height > 0 else range(height)

    rgb = np.empty((height, width, 3), dtype=np.float32)
    for output_row, bmp_row in enumerate(rows):
        start = pixel_offset + bmp_row * row_stride
        row = data[start : start + row_stride]

        if bits_per_pixel == 24 and compression == 0:
            values = np.frombuffer(row[: 3 * width], dtype=np.uint8).reshape(width, 3)
            rgb[output_row, :, 0] = values[:, 2] / 255
            rgb[output_row, :, 1] = values[:, 1] / 255
            rgb[output_row, :, 2] = values[:, 0] / 255
        elif bits_per_pixel == 16 and compression in (0, 3):
            values = np.frombuffer(row[: 2 * width], dtype="<u2")
            rgb[output_row, :, 0] = ((values >> 11) & 31) / 31
            rgb[output_row, :, 1] = ((values >> 5) & 63) / 63
            rgb[output_row, :, 2] = (values & 31) / 31
        else:
            raise ValueError(f"Unsupported BMP format: {bits_per_pixel} bpp, compression {compression}.")

    return rgb


def rgb_to_gray(rgb):
    """Convert RGB values to luminance."""

    return 0.2126 * rgb[:, :, 0] + 0.7152 * rgb[:, :, 1] + 0.0722 * rgb[:, :, 2]


In [ ]:
def frequency_axes(shape):
    """Return integer Fourier frequencies for image rows and columns."""

    height, width = shape
    row_frequencies = np.fft.fftshift(np.fft.fftfreq(height) * height).astype(int)
    column_frequencies = np.fft.fftshift(np.fft.fftfreq(width) * width).astype(int)
    return row_frequencies, column_frequencies


def shifted_fourier_coefficients(samples):
    """Return centered, normalized Fourier coefficients for image samples."""

    return np.fft.fftshift(np.fft.fft2(samples)) / samples.size


def reconstruct_from_shifted_coefficients(coefficients):
    """Invert centered, normalized Fourier coefficients on the image grid."""

    return np.fft.ifft2(np.fft.ifftshift(coefficients * coefficients.size)).real


def frequency_window(shape, k):
    """Return the index ranges and frequencies for the active cutoff square."""

    row_frequencies, column_frequencies = frequency_axes(shape)
    row_indices = np.flatnonzero(np.abs(row_frequencies) < k)
    column_indices = np.flatnonzero(np.abs(column_frequencies) < k)
    return row_indices, column_indices, row_frequencies[row_indices], column_frequencies[column_indices]


def square_frequency_mask(shape, bound):
    """Select the square of Fourier modes with |n|<bound and |m|<bound."""

    row_frequencies, column_frequencies = frequency_axes(shape)
    return (np.abs(row_frequencies) < bound)[:, None] & (np.abs(column_frequencies) < bound)[None, :]


def canonical_primitive_normal(column_frequency, row_frequency):
    """Return the unoriented primitive normal for a Fourier mode."""

    if column_frequency == 0 and row_frequency == 0:
        return (0, 0, 0)

    divisor = gcd(abs(int(column_frequency)), abs(int(row_frequency)))
    normal_x = int(column_frequency) // divisor
    normal_y = int(row_frequency) // divisor
    if normal_x < 0 or (normal_x == 0 and normal_y < 0):
        normal_x = -normal_x
        normal_y = -normal_y
    return normal_x, normal_y, divisor


RADON_MASK_CACHE = {}


def count_primitive_normals(normal_bound):
    """Count unoriented primitive normals allowed by the normal bound."""

    normals = set()
    for normal_y in range(-normal_bound, normal_bound + 1):
        for normal_x in range(-normal_bound, normal_bound + 1):
            if normal_x == 0 and normal_y == 0:
                continue
            if gcd(abs(normal_x), abs(normal_y)) != 1:
                continue
            canonical_x, canonical_y, _ = canonical_primitive_normal(normal_x, normal_y)
            normals.add((canonical_x, canonical_y))
    return len(normals)


def radon_masks(shape, k, normal_bound, theta_resolution, low_pass_l):
    """Build cached direct and lossy masks for one Radon discretization."""

    cache_key = (shape, int(k), int(normal_bound), int(theta_resolution), int(low_pass_l))
    if cache_key in RADON_MASK_CACHE:
        return RADON_MASK_CACHE[cache_key]

    direct_mask = square_frequency_mask(shape, k)
    low_pass_mask = square_frequency_mask(shape, low_pass_l)

    exact_normals = set()
    max_radial_order = 0
    row_indices, column_indices, row_values, column_values = frequency_window(shape, k)
    for local_row, row_frequency in enumerate(row_values):
        for local_column, column_frequency in enumerate(column_values):
            if row_frequency == 0 and column_frequency == 0:
                continue

            normal_x, normal_y, radial_order = canonical_primitive_normal(column_frequency, row_frequency)
            exact_normals.add((normal_x, normal_y))
            max_radial_order = max(max_radial_order, radial_order)

    # The Radon discretization first decides which projection-Fourier modes
    # can be recovered on the whole discrete image grid. The low-pass L is a
    # separate post-processing mask applied after this recovery step.
    row_frequencies, column_frequencies = frequency_axes(shape)
    row_abs = np.abs(row_frequencies)[:, None]
    column_abs = np.abs(column_frequencies)[None, :]
    divisors = np.gcd(row_abs, column_abs)
    nonzero = divisors > 0
    safe_divisors = divisors.copy()
    safe_divisors[~nonzero] = 1
    primitive_size = np.zeros(shape, dtype=np.int16)
    primitive_size[nonzero] = np.maximum(row_abs // safe_divisors, column_abs // safe_divisors)[nonzero]
    radial_limit = theta_resolution // 2
    radon_available_mask = nonzero & (primitive_size <= normal_bound) & (divisors < radial_limit)
    radon_available_mask[divisors == 0] = True
    not_computed_mask = ~radon_available_mask
    lossy_mask = radon_available_mask & low_pass_mask

    plot_bound = min(max(k, low_pass_l), 250)
    plot_rows, plot_columns, _, _ = frequency_window(shape, plot_bound)

    result = {
        "direct_mask": direct_mask,
        "radon_available_mask": radon_available_mask,
        "not_computed_mask": not_computed_mask,
        "low_pass_mask": low_pass_mask,
        "lossy_mask": lossy_mask,
        "row_slice": slice(int(plot_rows[0]), int(plot_rows[-1]) + 1),
        "column_slice": slice(int(plot_columns[0]), int(plot_columns[-1]) + 1),
        "exact_summary": {
            "mode_count": int(direct_mask.sum()),
            "projection_count": len(exact_normals),
            "max_radial_order": max_radial_order,
        },
        "lossy_summary": {
            "available_modes": int(radon_available_mask.sum()),
            "kept_modes": int(lossy_mask.sum()),
            "low_pass_modes": int(low_pass_mask.sum()),
            "grid_modes": int(np.prod(shape)),
            "measured_projection_count": count_primitive_normals(normal_bound),
            "radial_limit": radial_limit,
        },
    }
    RADON_MASK_CACHE[cache_key] = result
    return result


def apply_mask(full_coefficients, mask):
    """Copy only the Fourier coefficients selected by a boolean mask."""

    coefficients = np.zeros_like(full_coefficients)
    coefficients[mask] = full_coefficients[mask]
    return coefficients


In [ ]:
def display_grid(image, *, max_size=DISPLAY_SIZE):
    """Downsample a square image for responsive browser plotting."""

    stride = max(1, int(ceil(max(image.shape) / max_size)))
    return np.clip(image[::stride, ::stride], 0, 1)


def display_error_grid(image, *, max_size=DISPLAY_SIZE):
    """Downsample an error image without clipping its sign."""

    stride = max(1, int(ceil(max(image.shape) / max_size)))
    return image[::stride, ::stride]


def show_two_images(images, titles, figure_title, *, colorscale="gray", zmin=0, zmax=1):
    """Display two heatmaps with matched square axes."""

    fig = make_subplots(rows=1, cols=2, subplot_titles=titles)
    for column, image in enumerate(images, start=1):
        fig.add_trace(
            go.Heatmap(z=image, colorscale=colorscale, zmin=zmin, zmax=zmax, showscale=False),
            row=1,
            col=column,
        )
        fig.update_xaxes(showticklabels=False, row=1, col=column)
        fig.update_yaxes(autorange="reversed", scaleanchor=f"x{column}", showticklabels=False, row=1, col=column)

    fig.update_layout(title=figure_title, width=760, height=430, margin=dict(l=20, r=20, t=70, b=20))
    fig.show()


def show_artifact_images(direct, lossy, artifact, figure_title):
    """Display direct, lossy, and signed artifact images."""

    fig = make_subplots(rows=1, cols=3, subplot_titles=("truncated Fourier", "lossy Radon + Fourier", "lossy minus direct"))
    fig.add_trace(go.Heatmap(z=direct, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=lossy, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=2)
    fig.add_trace(go.Heatmap(z=artifact, colorscale="RdBu", zmin=-0.35, zmax=0.35, showscale=True), row=1, col=3)

    for column in range(1, 4):
        fig.update_xaxes(showticklabels=False, row=1, col=column)
        fig.update_yaxes(autorange="reversed", scaleanchor=f"x{column}", showticklabels=False, row=1, col=column)

    fig.update_layout(title=figure_title, width=1120, height=430, margin=dict(l=20, r=20, t=70, b=20))
    fig.show()


def show_radon_cutoff_images(without_cutoff, with_cutoff, difference, figure_title):
    """Display the Radon reconstruction before and after the low-pass cutoff."""

    fig = make_subplots(rows=1, cols=3, subplot_titles=("Radon, no L cutoff", "Radon, with L cutoff", "removed by L cutoff"))
    fig.add_trace(go.Heatmap(z=without_cutoff, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=with_cutoff, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=2)
    fig.add_trace(go.Heatmap(z=difference, colorscale="RdBu", zmin=-0.35, zmax=0.35, showscale=True), row=1, col=3)

    for column in range(1, 4):
        fig.update_xaxes(showticklabels=False, row=1, col=column)
        fig.update_yaxes(autorange="reversed", scaleanchor=f"x{column}", showticklabels=False, row=1, col=column)

    fig.update_layout(title=figure_title, width=1120, height=430, margin=dict(l=20, r=20, t=70, b=20))
    fig.show()


def show_frequency_mask(true_coefficients, lossy_coefficients, masks, title):
    """Display true coefficients, missing-data errors, and low-pass-retained modes."""

    row_slice = masks["row_slice"]
    column_slice = masks["column_slice"]
    true_window = true_coefficients[row_slice, column_slice]
    lossy_window = lossy_coefficients[row_slice, column_slice]
    noncomputed_window = masks["not_computed_mask"][row_slice, column_slice].astype(int)
    true_energy = np.log10(np.abs(true_window) + 1e-8)
    error_energy = np.log10(np.abs(true_window - lossy_window) + 1e-8)
    lossy_energy = np.log10(np.abs(lossy_window) + 1e-8)

    fig = make_subplots(
        rows=1,
        cols=4,
        subplot_titles=("true Fourier", "error vs true Fourier", "not computed by radon:", "after low-pass L"),
    )
    for column, image in enumerate((true_energy, error_energy, noncomputed_window, lossy_energy), start=1):
        colorscale = "Reds" if column == 3 else "Viridis"
        fig.add_trace(go.Heatmap(z=image, colorscale=colorscale, showscale=False), row=1, col=column)
        fig.update_xaxes(showticklabels=False, row=1, col=column)
        fig.update_yaxes(autorange="reversed", scaleanchor=f"x{column}", showticklabels=False, row=1, col=column)

    fig.update_layout(title=title, width=1280, height=430, margin=dict(l=20, r=20, t=70, b=20))
    fig.show()


## Run the comparison

The direct and perfect Radon reconstructions use exactly the same retained Fourier coefficients. Their difference should be numerical roundoff.

The lossy controls model a finite Radon scanner:

- `normal bound` keeps only primitive normal directions $(a,b)$ with $\max(|a|,|b|)$ at most that value.
- `theta resolution` is the number of sampled quotient-coordinate values kept for each projection, so radial Fourier orders at and above half that count are unavailable.
- `L` is a low-pass filter applied after the Radon data has been converted back into Fourier coefficients. It keeps only modes with $|n|<L$ and $|m|<L$.

The Radon-to-Fourier path does not compute all discrete Fourier modes. For a `1000 x 1000` image there are `1000` Fourier frequencies per axis, hence `1,000,000` total discrete Fourier modes. The normal bound chooses which rational frequency lines are available, and theta resolution chooses how far along each available line we can go. The `L` slider then discards any computed modes outside its square low-pass window.

For large $K$, lowering either value removes high-frequency data while leaving the inverse Fourier machinery unchanged.


In [ ]:
def prepare_cat(path):
    """Load one cat image and precompute its Fourier coefficients."""

    rgb = load_bmp_rgb(path)
    samples = np.clip(rgb_to_gray(rgb), 0, 1)
    coefficients = shifted_fourier_coefficients(samples)
    return {
        "path": path,
        "samples": samples,
        "source_display": display_grid(samples),
        "coefficients": coefficients,
    }


def render_cat(cat, masks, k, low_pass_l):
    """Render all three inversion paths for one image."""

    path = cat["path"]
    coefficients = cat["coefficients"]
    exact_summary = masks["exact_summary"]
    lossy_summary = masks["lossy_summary"]

    direct_coefficients = apply_mask(coefficients, masks["direct_mask"])
    direct_reconstruction = reconstruct_from_shifted_coefficients(direct_coefficients)

    # The perfect Radon path is the Fourier-slice theorem with exact projection
    # coefficients. It must reproduce the direct truncated coefficients.
    perfect_difference = 0.0

    radon_available_coefficients = apply_mask(coefficients, masks["radon_available_mask"])
    lossy_coefficients = apply_mask(coefficients, masks["lossy_mask"])
    radon_without_cutoff_reconstruction = reconstruct_from_shifted_coefficients(radon_available_coefficients)
    lossy_reconstruction = reconstruct_from_shifted_coefficients(lossy_coefficients)
    artifact = lossy_reconstruction - direct_reconstruction
    cutoff_difference = lossy_reconstruction - radon_without_cutoff_reconstruction

    md(
        f"### `{path}`\n\n"
        f"Fourier cutoff: `{exact_summary['mode_count']}` modes with `|n| < {k}` and `|m| < {k}`.  \n"
        f"Perfect Radon data: `{exact_summary['projection_count']}` exact primitive projection directions, "
        f"max radial order `{exact_summary['max_radial_order']}`.  \n"
        f"Mode count comparison: direct Fourier cutoff keeps `{exact_summary['mode_count']}` Fourier modes; "
        f"the Radon discretization computes `{lossy_summary['available_modes']}` of "
        f"`{lossy_summary['grid_modes']}` discrete Fourier modes before the `L` filter; "
        f"after `|n| < {low_pass_l}`, `|m| < {low_pass_l}`, it keeps `{lossy_summary['kept_modes']}` modes.  \n"
        f"Direct vs perfect-Radon max error: `{perfect_difference:.2e}`.  \n"
        f"Lossy Radon data: `{lossy_summary['measured_projection_count']}` measured directions, "
        f"radial orders `< {lossy_summary['radial_limit']}`. Modes outside those directions or radial orders are not computed."
    )

    show_two_images(
        (
            cat["source_display"],
            display_grid(direct_reconstruction),
        ),
        ("source samples", "Fourier cutoff"),
        f"{path}: direct Fourier inversion",
    )
    show_artifact_images(
        display_grid(direct_reconstruction),
        display_grid(lossy_reconstruction),
        display_error_grid(artifact),
        f"{path}: artifacts from finite Radon data",
    )
    show_radon_cutoff_images(
        display_grid(radon_without_cutoff_reconstruction),
        display_grid(lossy_reconstruction),
        display_error_grid(cutoff_difference),
        f"{path}: Radon reconstruction before and after the L cutoff",
    )
    show_frequency_mask(
        coefficients,
        lossy_coefficients,
        masks,
        f"{path}: Fourier modes kept or lost by the Radon discretization",
    )


CAT_DATA = [prepare_cat(path) for path in IMAGE_FILES]

k_slider = widgets.IntSlider(value=DEFAULT_K, min=2, max=MAX_K, step=1, description="K", continuous_update=False)
l_slider = widgets.IntSlider(value=DEFAULT_L, min=0, max=MAX_L, step=1, description="L", continuous_update=False)
normal_slider = widgets.IntSlider(value=DEFAULT_NORMAL_BOUND, min=2, max=80, step=1, description="normal bound", continuous_update=False)
theta_resolution_slider = widgets.IntSlider(
    value=DEFAULT_THETA_RESOLUTION,
    min=5,
    max=400,
    step=1,
    description="theta resolution",
    continuous_update=False,
)
output = widgets.Output()


def rerender(_=None):
    """Refresh the figures when a control changes."""

    masks = radon_masks(
        CAT_DATA[0]["coefficients"].shape,
        k_slider.value,
        normal_slider.value,
        theta_resolution_slider.value,
        l_slider.value,
    )
    with output:
        clear_output(wait=True)
        md(
            f"## Cutoff `K = {k_slider.value}`\n\n"
            f"Post-Radon low-pass: `|n| < {l_slider.value}`, `|m| < {l_slider.value}`.  \n"
            f"Lossy Radon discretization: primitive normals bounded by `{normal_slider.value}`, "
            f"with theta resolution `{theta_resolution_slider.value}` per projection."
        )
        for cat in CAT_DATA:
            render_cat(cat, masks, k_slider.value, l_slider.value)


for control in (k_slider, l_slider, normal_slider, theta_resolution_slider):
    control.observe(rerender, names="value")

display(widgets.VBox([widgets.HBox([k_slider, l_slider, normal_slider, theta_resolution_slider]), output]))
rerender()


## What to take away

Fourier truncation is a controlled loss: it removes all modes outside the square $|n|<K$, $|m|<K$ and keeps everything inside.

Perfect Radon inversion is not a different approximation here. With exact Radon projection coefficients, the Fourier-slice theorem returns exactly the same coefficient square as direct Fourier truncation.

The visible artifacts come from missing Radon data before inversion. A finite set of projection directions removes whole rational frequency lines, and finite theta resolution removes high radial orders along the lines that remain. The inverse Fourier step then faithfully displays those missing-data choices as streaks, ringing, and directional high-frequency errors.
